# TensorFlow / Keras 核心 API 全流程示例（根据截图内容整理）

> **这份笔记做什么？**
>
> 把书上截图里的核心内容完整串起来，从"什么是层"到"怎么训练模型"，
> 一步步演示 Keras 的完整工作流程。
>
> 你会学到：
> 1. **层与模型的关系** — 层是积木，模型是搭好的城堡
> 2. **自定义层** — 怎么写 `build()` 和 `call()`
> 3. **自动形状推断** — 为什么不用手写输入维度
> 4. **Sequential vs 函数式 API** — 两种搭模型的方式
> 5. **Transformer 结构** — 多头注意力 + 残差连接
> 6. **compile / fit / evaluate / predict** — 训练四步曲
> 7. **常见损失函数和指标** — 二分类、多分类、回归各用什么

> 说明：
> - 为了保证 Notebook 可直接运行，这里主要使用**可控的随机数据 / 合成数据**。
> - 你可以先跑通整个 Notebook，再把数据替换成你自己的真实数据。
> - 注释尽量写详细，方便你学习与汇报。

In [ ]:
# =========================
# Cell 0：基础导入
# =========================
#
# 三个库各管一件事：
# - numpy：生成模拟数据、数组操作
# - tensorflow：深度学习框架，提供各种层、优化器、损失函数
# - keras：TensorFlow 的高层 API，更简洁易用
#
# tensorflow 和 keras 的关系：
# - TensorFlow 是"底层引擎"，负责张量运算、梯度计算、GPU 加速
# - Keras 是"高层接口"，把常见的层、模型、训练流程封装成简单 API
# - 你用 Keras 写代码，底层实际跑的还是 TensorFlow
#
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow 版本：{tf.__version__}")
print(f"Keras 版本：{keras.__version__}")

---
## Cell 1：层与模型的关系 — 深度学习的"积木哲学"

### 什么是层（Layer）？

层就是一个**数据处理模块**：
- 输入：一个或多个张量
- 输出：一个或多个张量
- 内部：可能有一些可训练的参数（权重 W、偏置 b）

比如：
- `Dense` 层：全连接层，做 `y = xW + b`
- `Conv2D` 层：卷积层，用小窗口扫描图像提取特征
- `LSTM` 层：循环层，处理序列数据（时间序列、文本）

### 什么是模型（Model）？

模型就是**很多层组合在一起的结构**：
- 简单模型：层按顺序堆叠（Sequential）
- 复杂模型：层之间有分支、跳跃连接、多输入多输出

### 关键理解

从代码角度看：
- `Layer` 类负责封装：
  - **状态（state）**：可训练参数，比如权重矩阵 W、偏置向量 b
  - **计算（computation）**：前向传播逻辑，"输入进来怎么变成输出"
- `Model` 类是 `Layer` 的子类，所以模型也可以像层一样被调用

### Keras 的标准工作流程

```
1. 定义层（Dense, Conv2D, LSTM...）
2. 把层组合成模型（Sequential 或函数式 API）
3. compile() — 配置优化器、损失函数、指标
4. fit() — 开始训练
5. evaluate() / predict() — 评估和预测
```

这就是整个深度学习的"流水线"，后面每个步骤都会详细展开。

---
## Cell 2：自定义一个最简单的 Dense 层

### 为什么要自定义层？

Keras 内置了很多层（Dense, Conv2D, LSTM...），但有时候你需要：
- 实现特殊的计算逻辑
- 把多个操作打包成一个模块
- 学习层的内部原理

### 自定义层的两个核心方法

书上代码清单 3-22 展示了关键：

| 方法 | 什么时候调用 | 做什么 |
|------|-------------|--------|
| `build(input_shape)` | 第一次收到输入时 | 创建权重参数 |
| `call(inputs)` | 每次前向传播 | 定义计算逻辑 |

### 为什么用 `build()` 而不是 `__init__()`？

这是 Keras 的精妙设计：
- `__init__()` 时你只知道"输出维度是多少"
- `build()` 时你才知道"输入维度是多少"
- 这样就不用手写 `input_dim`，让它自动推断

### Dense 层的数学本质

就是最简单的线性变换 + 激活函数：

$$\text{output} = \text{activation}(XW + b)$$

- X：输入矩阵，形状 (batch_size, input_dim)
- W：权重矩阵，形状 (input_dim, units)
- b：偏置向量，形状 (units,)
- activation：激活函数（relu, sigmoid, softmax...）

下面代码实现的就是这个公式。

In [ ]:
# =========================
# Cell 2：自定义 SimpleDense 层实现
# =========================

class SimpleDense(keras.layers.Layer):
    """
    自定义全连接层（Dense Layer）。
    
    这就是 Dense 层的本质实现，展示 Keras 层的核心机制。
    
    参数
    ----
    units : int
        输出维度（神经元个数）。
        比如 units=32，输出就是 32 维向量。
    
    activation : callable 或 None
        激活函数，例如 tf.nn.relu, tf.nn.sigmoid, tf.nn.softmax。
        如果为 None，输出就是纯线性变换结果。
    
    **kwargs : dict
        传给父类的其他参数，比如 name="my_layer"。
    
    示例
    ----
    >>> layer = SimpleDense(units=32, activation=tf.nn.relu)
    >>> output = layer(input_tensor)  # 第一次调用时自动 build
    
    数学公式
    -------
    output = activation(X @ W + b)
    
    其中：
    - X：输入，形状 (batch_size, input_dim)
    - W：权重，形状 (input_dim, units)
    - b：偏置，形状 (units,)
    """
    
    def __init__(self, units, activation=None, **kwargs):
        """初始化层，只记住配置，不创建参数。"""
        super().__init__(**kwargs)
        self.units = units          # 输出维度
        self.activation = activation  # 激活函数
    
    def build(self, input_shape):
        """
        创建权重参数。
        
        这个方法会在第一次调用层时自动执行。
        Keras 通过 input_shape 知道输入长什么样，然后创建匹配的权重。
        
        参数
        ----
        input_shape : tuple
            输入张量的形状，比如 (batch_size, 784)。
            input_shape[-1] 就是输入特征维度（784）。
        
        为什么要这样设计？
        ----------------
        因为初始化时你只知道输出维度（units），
        只有看到输入后才知道权重矩阵该多大。
        
        比如输入是 (batch, 784)，输出是 32 维，
        那权重矩阵必须是 (784, 32)，这由 input_shape 决定。
        """
        input_dim = input_shape[-1]  # 取最后一维作为输入维度
        
        # add_weight() 是 Keras 创建可训练参数的标准方式
        # 相比 tf.Variable，它会自动处理：
        # - 正则化
        # - 约束条件
        # - 模型保存/加载
        
        # W — 权重矩阵
        # 形状：(input_dim, units)
        # 含义：每个输入特征到每个输出神经元的连接权重
        self.W = self.add_weight(
            shape=(input_dim, self.units),
            initializer="random_normal",  # 随机初始化
            trainable=True,               # 可训练（会被梯度更新）
            name="kernel"                 # 名称（调试用）
        )
        
        # b — 偏置向量
        # 形状：(units,)
        # 含义：每个输出神经元的偏置值
        self.b = self.add_weight(
            shape=(self.units,),
            initializer="zeros",  # 通常初始化为 0
            trainable=True,
            name="bias"
        )
    
    def call(self, inputs):
        """
        定义前向传播逻辑：这一层拿到输入后做什么。
        
        参数
        ----
        inputs : tf.Tensor
            输入张量，形状 (batch_size, input_dim)。
        
        返回
        ----
        output : tf.Tensor
            输出张量，形状 (batch_size, units)。
        
        计算过程
        -------
        1. y = inputs @ W + b  （线性变换）
        2. 如果有激活函数，y = activation(y)
        3. 返回 y
        """
        # 线性变换：y = xW + b
        # tf.matmul：矩阵乘法
        # + b：广播加法，b 的形状 (units,) 自动扩展到 (batch, units)
        y = tf.matmul(inputs, self.W) + self.b
        
        # 如果设置了激活函数，就应用它
        if self.activation is not None:
            y = self.activation(y)
        
        return y

In [ ]:
# =========================
# Cell 3：像函数一样调用层
# =========================
#
# Keras 层的一大特点：可以像函数一样调用
# layer = SimpleDense(32)
# output = layer(input)  # 就像调用一个函数
#
# 第一次调用时会发生什么？
# 1. Keras 发现这个层还没有参数（built=False）
# 2. 自动调用 build(input_shape) 创建权重
# 3. 然后执行 call(inputs) 做前向计算
# 4. 之后再次调用时，直接执行 call()
#

# 创建一个层实例
# units=32：输出 32 维
# activation=tf.nn.relu：用 ReLU 激活函数
my_dense = SimpleDense(units=32, activation=tf.nn.relu)

# 构造一个测试输入
# 形状 (2, 784)：batch 中有 2 个样本，每个样本 784 个特征
# 这类似 MNIST 数据：28x28 图像展平成 784 维向量
input_tensor = tf.ones(shape=(2, 784))

# 第一次调用
# 此时 build() 会自动执行，创建形状为 (784, 32) 的权重矩阵
output_tensor = my_dense(input_tensor)

print(f"输入形状：{input_tensor.shape}")
print(f"  含义：{input_tensor.shape[0]} 个样本，每个 {input_tensor.shape[1]} 个特征")
print(f"\n输出形状：{output_tensor.shape}")
print(f"  含义：{output_tensor.shape[0]} 个样本，每个 {output_tensor.shape[1]} 个特征（变了！）")

# 查看这一层内部有哪些参数
print("\n层中的权重：")
for var in my_dense.weights:
    print(f"  {var.name}: 形状 {var.shape}")

# 计算参数总数
total_params = sum(var.numpy().size for var in my_dense.weights)
print(f"\n总参数量：{total_params} 个")
print(f"  = W: {784 * 32} 个（784×32）")
print(f"  + b: {32} 个")
print(f"  = {784 * 32 + 32} 个")

---
### 维度变化图解

```
输入 X                    权重 W                 输出 Y
(batch, 784)              (784, 32)              (batch, 32)
┌─────────────┐           ┌──────┐               ┌──────┐
│ 2 × 784     │    @      │784×32│      =        │ 2 × 32│
│             │           │      │               │      │
└─────────────┘           └──────┘               └──────┘
```

**关键理解**：
- 矩阵乘法的维度规则：`(a, b) @ (b, c) = (a, c)`
- 784 维输入 → 32 维输出，中间需要 (784, 32) 的权重矩阵
- batch 维度（2）保持不变，因为它是"有多少个样本"

**权重参数数量**：
- W：784 × 32 = 25,088 个
- b：32 个
- 总计：25,120 个参数

这就是为什么 Dense 层参数量 = `输入维度 × 输出维度 + 输出维度`

---
## Cell 4：为什么要用 `build()`？—— 自动形状推断的精妙之处

### 问题：如果不用 `build()` 会怎样？

假设你在 `__init__()` 里创建权重，那你必须这样写：

```python
# 不灵活的写法
layer = Dense(input_dim=784, output_dim=32)  # 必须指定 input_dim
```

问题来了：
- 改了上一层的输出，下一层的 `input_dim` 也要跟着改
- 层数多了，改一个地方要动一堆代码
- 容易出错，维度不匹配报错很难排查

### Keras 的解决方案：延迟构建

```python
# Keras 风格
layer = Dense(32)  # 只指定输出维度
output = layer(input)  # 第一次调用时自动推断输入维度
```

**流程**：
1. `__init__()` 只记住配置（units=32），不创建参数
2. 第一次 `layer(input)` 时，Keras 发现还没有参数
3. 自动调用 `build(input.shape)`，根据输入形状创建权重
4. 之后就直接用 `call()` 计算了

### 这样做的好处

- **代码简洁**：不用到处写 `input_dim`
- **不易出错**：维度自动匹配
- **灵活组合**：可以随意换层的位置

这就是书上说的"自动推断形状（动态构建层）"的核心思想。

---
## Cell 5：对比实验 — 一个"笨"的 NaiveDense

为了让你深刻理解 Keras 的自动构建机制有多好用，
我们先实现一个"笨版本"，看看它有多麻烦。

In [ ]:
# =========================
# Cell 5：一个"笨"的 NaiveDense 实现
# =========================
#
# 这个版本必须在初始化时就指定 input_size 和 output_size
# 对比 Keras 的 Dense 层，你会发现它有多不方便
#

class NaiveDense:
    """
    一个不够灵活的全连接层实现。
    
    问题：必须在初始化时指定输入和输出维度。
    后果：一旦上一层的输出维度变了，这一行也要改。
    
    参数
    ----
    input_size : int
        输入特征维度（必须手写）。
    
    output_size : int
        输出特征维度。
    
    activation : callable 或 None
        激活函数。
    """
    
    def __init__(self, input_size, output_size, activation=None):
        """初始化时就要知道输入维度，不够灵活。"""
        self.input_size = input_size
        self.output_size = output_size
        self.activation = activation
        
        # 直接在初始化时创建参数
        # 问题：如果 input_size 搞错了，权重形状就错了
        self.W = tf.Variable(
            tf.random.normal(shape=(input_size, output_size)),
            trainable=True
        )
        self.b = tf.Variable(
            tf.zeros(shape=(output_size,)),
            trainable=True
        )
    
    def __call__(self, inputs):
        """
        前向传播。
        
        参数
        ----
        inputs : tf.Tensor
            输入张量，形状必须匹配 input_size，否则会报错。
        """
        y = tf.matmul(inputs, self.W) + self.b
        if self.activation is not None:
            y = self.activation(y)
        return y

In [ ]:
# =========================
# Cell 6：用 NaiveDense 手工拼一个"顺序模型"
# =========================
#
# 看看用笨版本有多麻烦：
# 每一层的 input_size 都要手写，而且必须和上一层的 output_size 对上
#

naive_layers = [
    NaiveDense(input_size=784, output_size=32, activation=tf.nn.relu),
    #             ↑ 必须写        ↑ 下一层这里也要改
    NaiveDense(input_size=32, output_size=64, activation=tf.nn.relu),
    #             ↑ 必须等于上一层的 output_size
    NaiveDense(input_size=64, output_size=32, activation=tf.nn.relu),
    NaiveDense(input_size=32, output_size=10, activation=tf.nn.softmax),
]

# 手工写前向传播循环
x = tf.ones((2, 784))
for layer in naive_layers:
    x = layer(x)

print(f"NaiveSequential 最终输出形状：{x.shape}")
print("\n问题：如果我想把第二层改成 output_size=128，")
print("那第三层的 input_size 也要跟着改成 128，")
print("第四层又得改... 改一个地方，动一堆代码。")

---
### 问题总结

用 NaiveDense 的痛苦：

```
第1层：input_size=784, output_size=32
第2层：input_size=32,  output_size=64   ← output_size 改成 128？
第3层：input_size=64,  output_size=32   ← input_size 也要改成 128！
第4层：input_size=32,  output_size=10   ← input_size 又要改成 128！
```

**一改全改**，这就是没有自动形状推断的下场。

Keras 的 Dense 层完美解决这个问题：
```python
model = Sequential([
    Dense(32),   # 不用写 input_dim
    Dense(64),   # 自动知道输入是 32
    Dense(32),   # 自动知道输入是 64
    Dense(10),   # 自动知道输入是 32
])
```

---
## Cell 7：Keras 自动构建机制的内幕

### Layer.__call__() 的简化逻辑

Keras 的 `Layer` 类在 `__call__()` 里做了这件事：

```python
def __call__(self, inputs):
    # 第一次调用时，如果还没 build，就先 build
    if not self.built:
        self.build(inputs.shape)  # 根据输入形状创建参数
        self.built = True         # 标记为已构建
    
    # 然后执行真正的前向传播
    return self.call(inputs)
```

### 执行流程图

```
第一次调用 layer(input)
         │
         ▼
   built == False?
         │
    ┌────┴────┐
    │ Yes     │ No
    ▼         ▼
build()    call()
    │         │
    ▼         │
call() ◄─────┘
    │
    ▼
  输出
```

### 这就是 Keras "魔法"的真相

- 你只管写 `Dense(32)`
- Keras 替你处理所有维度匹配
- 第一次调用时自动完成构建

这个设计贯穿整个 Keras API，让代码既简洁又不易出错。

---
## Cell 8：用自定义层搭建 Sequential 模型

有了自动形状推断，代码变得超级简洁。
这里用我们自己写的 `SimpleDense` 搭一个完整的模型。

In [ ]:
# =========================
# Cell 8：用自定义层搭建 Sequential 模型
# =========================
#
# Sequential：最简单的模型类型，层按顺序堆叠
#
# 对比 NaiveDense 的版本：
# - 不用写 input_size
# - 不用手工写 for 循环
# - 一行代码定义整个模型
#

model_from_custom_layers = keras.Sequential([
    SimpleDense(32, activation=tf.nn.relu),    # 输出 32 维
    SimpleDense(64, activation=tf.nn.relu),    # 输出 64 维
    SimpleDense(32, activation=tf.nn.relu),    # 输出 32 维
    SimpleDense(10, activation=tf.nn.softmax)  # 输出 10 维（分类问题）
], name="my_custom_model")

# 第一次喂数据时，每层的 build() 自动执行
x = tf.ones((4, 784))  # 4 个样本，每个 784 维
y = model_from_custom_layers(x)

print(f"输入形状：{x.shape}")
print(f"输出形状：{y.shape}")
print(f"\n模型结构：")
model_from_custom_layers.summary()

# summary() 输出解读：
# - Layer (type)：每层的名字和类型
# - Output Shape：每层输出的形状
# - Param #：参数数量
# - 最下面是总参数量

---
## Cell 9：从层到模型 — Sequential vs 函数式 API

### Sequential 的局限

Sequential 只能处理"一条线"的结构：

```
Input → Layer1 → Layer2 → Layer3 → Output
```

但很多先进模型的结构更复杂，比如：

- **残差连接（ResNet）**：跳过某些层，把输入直接加到输出上
- **多输入多输出**：比如图像 + 文本同时输入，输出分类 + 描述
- **Inception 结构**：同一层有多种尺寸的卷积，然后拼接

这些就没法用 Sequential 写了。

### 函数式 API（Functional API）

Keras 提供了更灵活的方式：

```python
# 函数式 API 风格
inputs = keras.Input(shape=(784,))
x = layers.Dense(64)(inputs)
x = layers.Dense(64)(x)
outputs = layers.Dense(10)(x)

model = keras.Model(inputs, outputs)
```

看起来像函数调用，所以叫"函数式 API"。

下面用两种方式搭建同样的模型，对比一下。

---
## Cell 10：函数式 API 示例 — 简单 MLP

MLP = Multi-Layer Perceptron = 多层感知机 = 全连接网络

这是一个经典的网络结构：输入 → 两层隐藏层 → 输出

In [ ]:
# =========================
# Cell 10：函数式 API 搭建 MLP
# =========================
#
# 函数式 API 的核心思想：
# 1. 先定义输入节点：keras.Input()
# 2. 每一层像函数一样调用，传入上一层的输出
# 3. 最后用 Model 类把输入输出打包成模型
#

# 1️⃣ 定义输入
# shape=(784,) 表示每个样本是 784 维向量
# 不写 batch 维度，因为 batch 大小可以变
inputs = keras.Input(shape=(784,), name="features")

# 2️⃣ 定义层和连接
# 注意语法：layers.Dense(64)(inputs)
#            ↑ 创建层         ↑ 调用层，传入输入
x = layers.Dense(64, activation="relu", name="hidden_1")(inputs)
x = layers.Dense(64, activation="relu", name="hidden_2")(x)
outputs = layers.Dense(10, activation="softmax", name="predictions")(x)

# 3️⃣ 创建模型
# 把"从 inputs 到 outputs 的计算图"打包成一个模型对象
functional_mlp = keras.Model(inputs=inputs, outputs=outputs, name="functional_mlp")

# 查看模型结构
functional_mlp.summary()

# summary() 的关键信息：
# ┌──────────────────────────────────────────────────┐
# │ Layer (type)           Output Shape    Param #   │
# ├──────────────────────────────────────────────────┤
# │ features (InputLayer)  [(None, 784)]    0        │  ← 输入层
# │ hidden_1 (Dense)       (None, 64)       50240    │  ← 784×64+64=50240
# │ hidden_2 (Dense)       (None, 64)       4160     │  ← 64×64+64=4160
# │ predictions (Dense)    (None, 10)       650      │  ← 64×10+10=650
# └──────────────────────────────────────────────────┘
# Total params: 55,050

---
## Cell 11：Transformer 风格残差块 — 函数式 API 的真正威力

Transformer 是近年来最重要的神经网络结构之一（GPT、BERT 都基于它）。
它的核心组件包括：

### 关键组件

| 组件 | 作用 |
|------|------|
| `MultiHeadAttention` | 多头注意力机制，让模型"关注"序列的不同位置 |
| `LayerNormalization` | 层归一化，稳定训练 |
| 前馈网络 `Dense → Dense` | 对每个位置独立做非线性变换 |
| 残差连接 `x + layer(x)` | 缓解梯度消失，让深层网络可训练 |

### 残差连接是什么？

普通网络：`output = layer(input)`
残差网络：`output = input + layer(input)`

```
        ┌─────────────┐
        │    input    │
        └──────┬──────┘
               │
        ┌──────┴──────┐
        │             │
        ▼             ▼
   ┌─────────┐       │
   │  layer  │       │
   └────┬────┘       │
        │            │
        ▼            │
   ┌─────────┐       │
   │  output │ ◄─────┘  ← 把原始输入加回来
   └─────────┘
```

**好处**：梯度可以直接"跳过"某些层传回去，深层网络也能训练了。

下面实现一个最小的 Transformer Encoder Block。

In [ ]:
# =========================
# Cell 11：最小 Transformer Encoder Block 实现
# =========================
#
# Transformer 的核心：自注意力 + 残差连接 + 层归一化
#
# 自注意力（Self-Attention）：
#   让序列中每个位置都能"看到"其他所有位置
#   比如处理句子时，"它"这个词能关联到"猫"
#

class TransformerEncoder(layers.Layer):
    """
    Transformer Encoder Block。
    
    结构：
        Input
          │
          ├──────────────────┐
          ▼                  │
    MultiHeadAttention        │
          │                  │
          ▼                  │
        Add ◄────────────────┘  ← 残差连接
          │
          ▼
    LayerNormalization
          │
          ├──────────────────┐
          ▼                  │
    Dense(relu) → Dense      │
          │                  │
          ▼                  │
        Add ◄────────────────┘  ← 残差连接
          │
          ▼
    LayerNormalization
          │
        Output
    
    参数
    ----
    embed_dim : int
        嵌入维度，也是输入/输出的特征维度。
    
    dense_dim : int
        前馈网络中间层的维度，通常比 embed_dim 大。
    
    num_heads : int
        注意力头的数量，embed_dim 必须能被 num_heads 整除。
    """
    
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        
        # 多头注意力层
        # num_heads：几个注意力头
        # key_dim：每个头的维度（这里简化用 embed_dim）
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )
        
        # 前馈网络：先放大，再缩回原尺寸
        # 这是一个"瓶颈"结构：embed_dim → dense_dim → embed_dim
        self.dense_proj = keras.Sequential([
            layers.Dense(dense_dim, activation="relu"),  # 放大
            layers.Dense(embed_dim)                       # 缩回
        ])
        
        # 两个层归一化
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
    
    def call(self, inputs):
        """
        前向传播。
        
        参数
        ----
        inputs : tf.Tensor
            输入序列，形状 (batch_size, sequence_length, embed_dim)。
        
        返回
        ----
        output : tf.Tensor
            输出序列，形状与输入相同。
        """
        # 1️⃣ 自注意力
        # attention(query, value) 这里 query=value=inputs
        # 意思是：用序列自己和自己做注意力
        attention_output = self.attention(inputs, inputs)
        
        # 2️⃣ 第一次残差连接 + 层归一化
        # inputs + attention_output：残差连接，把原始输入加回来
        # LayerNorm：归一化，稳定训练
        proj_input = self.layernorm_1(inputs + attention_output)
        
        # 3️⃣ 前馈网络
        proj_output = self.dense_proj(proj_input)
        
        # 4️⃣ 第二次残差连接 + 层归一化
        return self.layernorm_2(proj_input + proj_output)

In [ ]:
# =========================
# Cell 12：测试 TransformerEncoder
# =========================

# 构造一个假序列数据
# 形状：(batch_size, sequence_length, embed_dim)
# batch_size = 2：2 个样本
# sequence_length = 8：每个样本 8 个 token
# embed_dim = 32：每个 token 用 32 维向量表示
dummy_sequence = tf.random.normal(shape=(2, 8, 32))

# 创建 Transformer Encoder Block
transformer_block = TransformerEncoder(
    embed_dim=32,     # 输入/输出维度
    dense_dim=64,     # 前馈网络中间层维度
    num_heads=4       # 4 个注意力头
)

# 前向传播
transformer_output = transformer_block(dummy_sequence)

print(f"输入序列形状：{dummy_sequence.shape}")
print(f"输出序列形状：{transformer_output.shape}")
print("\n输入和输出形状相同，这是 Transformer 的特点：")
print("  - 序列长度不变（8 → 8）")
print("  - 特征维度不变（32 → 32）")
print("  - 但每个位置的表示融合了其他位置的信息")

---
### Transformer 结构总结

```
输入序列 (batch, 8, 32)
        │
        ▼
┌───────────────────────────────────┐
│     MultiHeadAttention            │
│     (4 heads, key_dim=32)         │
└───────────────────────────────────┘
        │
        ▼
┌───────────────────────────────────┐
│  Add & LayerNorm                  │  ← 残差连接：input + attention
│  (inputs + attention_output)      │
└───────────────────────────────────┘
        │
        ▼
┌───────────────────────────────────┐
│  Feed-Forward Network             │
│  Dense(64, relu) → Dense(32)      │
└───────────────────────────────────┘
        │
        ▼
┌───────────────────────────────────┐
│  Add & LayerNorm                  │  ← 残差连接：input + ffn
└───────────────────────────────────┘
        │
        ▼
输出序列 (batch, 8, 32)
```

这就是 GPT、BERT 等大模型的核心模块。

---
## Cell 13：编译模型 — `compile()` 配置训练规则

训练前要告诉模型三件事：

### 三大核心参数

| 参数 | 作用 | 常见选择 |
|------|------|----------|
| **optimizer** | 决定参数怎么更新 | SGD, Adam, RMSprop |
| **loss** | 衡量预测与真实标签的差异 | MSE, CrossEntropy |
| **metrics** | 训练时监控的指标 | Accuracy, Precision |

### 两种写法

**字符串形式**（简单快捷）：
```python
model.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
```

**对象形式**（可调参数）：
```python
model.compile(
    optimizer=keras.optimizers.RMSprop(learning_rate=1e-4),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=[keras.metrics.BinaryAccuracy()]
)
```

### compile() 做了什么？

- 不训练，只是配置
- 把优化器、损失函数、指标"绑定"到模型上
- 之后 `fit()` 时就知道该怎么训练了

---
## Cell 14：准备训练数据 — 二分类数据集

为了演示完整的训练流程，我们先造一个二分类数据集。

### 为什么是二分类？

最简单、最常见的任务类型：
- 邮件是不是垃圾邮件？
- 肿瘤是良性还是恶性？
- 用户会不会点击广告？

两种方法对比：
- `BinaryCrossentropy`：二分类交叉熵损失
- `BinaryAccuracy`：二分类准确率

In [ ]:
# =========================
# Cell 14：生成二分类数据集
# =========================
#
# 数据构造思路：
# 1. 随机生成输入特征（20 维向量）
# 2. 用一个线性变换生成"真实标签"
# 3. 加点噪声让问题更真实
#

import numpy as np

# 固定随机种子，保证每次运行结果一致
np.random.seed(42)
tf.random.set_seed(42)

num_samples = 2000   # 总样本数
num_features = 20    # 每个样本的特征数

# 随机生成输入特征
# 形状：(2000, 20)
# 每行是一个样本，每列是一个特征
inputs = np.random.randn(num_samples, num_features).astype("float32")

# 构造一个"真实"的线性分类边界
# 这就好像存在一条隐形的线，把两类点分开
true_w = np.random.randn(num_features, 1).astype("float32")

# 线性变换：logits = X @ W
# logits 是"未归一化的分数"，正值表示一类，负值表示另一类
logits = inputs @ true_w + 0.3 * np.random.randn(num_samples, 1).astype("float32")

# 转成 0/1 标签
# logits > 0 → 标签 = 1
# logits <= 0 → 标签 = 0
targets = (logits > 0).astype("float32")

print(f"输入数据形状：{inputs.shape}")
print(f"  含义：{inputs.shape[0]} 个样本，每个 {inputs.shape[1]} 个特征")
print(f"\n标签数据形状：{targets.shape}")
print(f"  含义：每个样本一个标签（0 或 1）")
print(f"\n标签分布：")
print(f"  类别 0：{(targets == 0).sum()} 个")
print(f"  类别 1：{(targets == 1).sum()} 个")
print(f"\n前 5 个标签：{targets[:5].flatten()}")

In [ ]:
# =========================
# Cell 15：构建二分类模型
# =========================
#
# 结构：输入(20) → Dense(32) → Dense(32) → Dense(1, sigmoid)
#
# 为什么最后一层是 sigmoid？
# - sigmoid 输出 0~1 之间的概率
# - 0.5 是分界线：> 0.5 预测为 1，< 0.5 预测为 0
#

binary_model = keras.Sequential([
    layers.Dense(32, activation="relu", name="hidden_1"),      # 20 → 32
    layers.Dense(32, activation="relu", name="hidden_2"),      # 32 → 32
    layers.Dense(1, activation="sigmoid", name="output")       # 32 → 1
], name="binary_classifier")

# 查看模型结构
binary_model.summary()

# 参数计算：
# hidden_1: 20×32 + 32 = 672
# hidden_2: 32×32 + 32 = 1056
# output:   32×1 + 1 = 33
# 总计: 672 + 1056 + 33 = 1761

In [ ]:
# =========================
# Cell 16：compile() — 字符串形式
# =========================
#
# 最简单的 compile 写法，用字符串指定优化器和损失函数
# Keras 会自动把它们转成对应的对象
#

binary_model.compile(
    optimizer="rmsprop",           # RMSprop 优化器
    loss="binary_crossentropy",    # 二分类交叉熵损失
    metrics=["accuracy"]           # 监控准确率
)

print("模型编译完成！")
print("\n配置说明：")
print("  - 优化器：rmsprop（自适应学习率）")
print("  - 损失函数：binary_crossentropy（二分类交叉熵）")
print("  - 监控指标：accuracy（准确率）")
print("\n现在模型准备好训练了，调用 fit() 即可开始。")


### 关于 `compile()` 的理解

你可以把 `compile()` 理解成：

> **“给模型配置训练规则”**

它本身不训练，只是把训练所需的关键组件配置好。

截图里提到的三大核心参数就是：

- `optimizer`
- `loss`
- `metrics`



## 12. 也可以传对象形式的优化器 / 损失 / 指标

截图里也提到这种更“面向对象”的写法，优点是方便细调参数，比如学习率。


In [ ]:

# =========================
# 13. compile()：传入对象形式
# =========================
binary_model_obj = keras.Sequential([
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid")
], name="binary_classifier_object_style")

binary_model_obj.compile(
    optimizer=keras.optimizers.RMSprop(learning_rate=1e-3),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=[keras.metrics.BinaryAccuracy()]
)

print("对象风格 compile 完成。")



## 13. 训练模型：`fit()`

截图中的 3.6.5 讲了 `fit()` 的几个关键参数：

- `inputs`：输入数据
- `targets`：对应标签
- `epochs`：训练轮数
- `batch_size`：批大小

典型写法：

```python
history = model.fit(
    inputs,
    targets,
    epochs=5,
    batch_size=128
)
```

返回值 `history` 是一个 `History` 对象，其中：

- `history.history` 是一个字典
- 字典的键是损失和各类指标名称
- 字典的值是每个 epoch 对应的数值列表


In [ ]:
# =========================
# Cell 17：fit() — 开始训练
# =========================
#
# fit() 的核心参数：
# - inputs：输入数据
# - targets：标签
# - epochs：训练多少轮（每轮遍历一次全部数据）
# - batch_size：每次更新用多少样本
# - verbose：打印信息的详细程度（0=静默，1=进度条，2=每轮一行）
#
# fit() 做的事情：
# 1. 把数据分成小批次（batch）
# 2. 对每个批次：前向传播 → 算 loss → 反向传播 → 更新参数
# 3. 重复 epochs 轮
# 4. 返回一个 History 对象，记录训练过程中的 loss 和 metrics
#

history = binary_model.fit(
    inputs,
    targets,
    epochs=5,         # 训练 5 轮
    batch_size=128,   # 每批 128 个样本
    verbose=1         # 打印进度条
)

# 2000 个样本，每批 128 个
# 每轮迭代次数 = 2000 / 128 ≈ 16 次
# 5 轮共 5 × 16 = 80 次参数更新

In [ ]:
# =========================
# Cell 18：查看训练历史
# =========================
#
# history.history 是一个字典，记录了训练过程中的所有指标
# 键名取决于 loss 和 metrics 的配置
#

print("history.history 的键：", list(history.history.keys()))
print("\n=== 训练历史 ===")
print(f"轮次    Loss      Accuracy")
print("-" * 35)
for i, (loss, acc) in enumerate(zip(
    history.history["loss"],
    history.history["accuracy"]
)):
    print(f"  {i+1}     {loss:.4f}    {acc:.4f}")

print("\n观察：")
print("  - Loss 应该逐渐下降")
print("  - Accuracy 应该逐渐上升")
print("  - 如果不这样，说明训练有问题")


### `history.history` 一般长什么样？

通常类似这样：

```python
{
    "loss": [...],
    "accuracy": [...]
}
```

如果你设置了验证集，那么通常还会多出：

```python
{
    "val_loss": [...],
    "val_accuracy": [...]
}
```

这也是截图中强调的重点。


---
## Cell 19：使用验证集 — `validation_data`

### 为什么需要验证集？

训练集：用来更新参数
验证集：用来检查模型是否"记住"了数据

**问题：过拟合**
- 训练集准确率 99%，验证集准确率 70%
- 说明模型"死记硬背"了训练数据，不会举一反三

### 正确做法

训练时同时监控：
- 训练集指标：`loss`, `accuracy`
- 验证集指标：`val_loss`, `val_accuracy`

如果训练集指标持续上升，验证集指标开始下降 → 过拟合了，该停了。

In [ ]:
# =========================
# Cell 20：划分训练集和验证集
# =========================
#
# 常见划分比例：80% 训练，20% 验证
# 这里 2000 个样本，取 400 个做验证集
#

num_val_samples = 400

# 打乱索引（防止数据有某种顺序性）
indices = np.arange(num_samples)
np.random.shuffle(indices)

# 按打乱后的索引重排数据
shuffled_inputs = inputs[indices]
shuffled_targets = targets[indices]

# 划分
val_inputs = shuffled_inputs[:num_val_samples]      # 前 400 个
val_targets = shuffled_targets[:num_val_samples]

train_inputs = shuffled_inputs[num_val_samples:]    # 后 1600 个
train_targets = shuffled_targets[num_val_samples:]

print(f"训练集：{train_inputs.shape[0]} 个样本")
print(f"验证集：{val_inputs.shape[0]} 个样本")
print(f"比例：{train_inputs.shape[0]/num_samples:.0%} / {val_inputs.shape[0]/num_samples:.0%}")

In [ ]:
# =========================
# Cell 21：带验证集的训练
# =========================
#
# 新参数：validation_data=(x_val, y_val)
# 每轮训练结束后，在验证集上评估一次，记录 val_loss 和 val_metrics
#

# 重新创建一个模型（之前的已经训练过了）
binary_model_with_val = keras.Sequential([
    layers.Dense(32, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid")
], name="binary_classifier_with_val")

# 对象形式的 compile（更灵活）
binary_model_with_val.compile(
    optimizer=keras.optimizers.RMSprop(learning_rate=1e-3),  # 可调学习率
    loss=keras.losses.BinaryCrossentropy(),
    metrics=[keras.metrics.BinaryAccuracy()]
)

# 训练，带验证集
history_with_val = binary_model_with_val.fit(
    train_inputs,
    train_targets,
    epochs=5,
    batch_size=16,
    validation_data=(val_inputs, val_targets),  # 验证集
    verbose=1
)

# 现在历史记录里会有：
# - loss, binary_accuracy（训练集）
# - val_loss, val_binary_accuracy（验证集）

In [ ]:

# =========================
# 18. 查看带验证集时的 history
# =========================
print(history_with_val.history.keys())
print(history_with_val.history)



### 为什么要看验证集？

因为训练集上的效果越来越好，并不代表模型真的泛化更好。

- **训练集指标高**：说明模型在“见过的数据”上拟合得不错
- **验证集指标高**：才更能说明模型对“没见过的数据”有泛化能力

这就是截图里对“验证损失 / 验证指标”的强调点。



## 15. 使用 `tf.data.Dataset` 进行训练

截图里提到：

- `fit()` 不仅能吃 NumPy 数组
- 还可以吃 `TensorFlow Dataset` 对象

这在大数据集训练中非常常见，因为它更利于：

- 批处理
- 打乱
- 预取
- 流式读取


In [ ]:

# =========================
# 19. 用 tf.data.Dataset 构造训练管道
# =========================
batch_size = 32

train_dataset = tf.data.Dataset.from_tensor_slices((train_inputs, train_targets))
train_dataset = train_dataset.shuffle(buffer_size=1024).batch(batch_size).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((val_inputs, val_targets))
val_dataset = val_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

print(train_dataset)
print(val_dataset)


In [ ]:

# =========================
# 20. 用 Dataset 训练
# =========================
binary_model_dataset = keras.Sequential([
    layers.Dense(32, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid")
], name="binary_classifier_dataset")

binary_model_dataset.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_dataset = binary_model_dataset.fit(
    train_dataset,
    epochs=3,
    validation_data=val_dataset,
    verbose=1
)


---
## Cell 22：评估模型 — `evaluate()`

训练完后，在独立测试集上评估最终效果。

### evaluate() vs fit()

| | fit() | evaluate() |
|---|-------|------------|
| 目的 | 训练，更新参数 | 评估，不更新参数 |
| 返回值 | History 对象 | [loss, metrics...] |
| 用途 | 训练阶段 | 测试阶段 |

### 典型用法

```python
loss, accuracy = model.evaluate(x_test, y_test)
```

In [ ]:
# =========================
# Cell 23：evaluate() 评估
# =========================
#
# 在验证集上评估最终效果
# 不更新参数，只计算 loss 和 metrics
#

results = binary_model_with_val.evaluate(
    val_inputs,
    val_targets,
    batch_size=128,
    verbose=1
)

print(f"\nevaluate() 返回结果：{results}")
print(f"  - results[0] = loss = {results[0]:.4f}")
print(f"  - results[1] = binary_accuracy = {results[1]:.4f}")
print(f"\n最终准确率：{results[1]*100:.1f}%")

---
## Cell 24：模型推理 — `model(x)` vs `predict()`

训练好的模型，用来对新数据做预测。

### 两种方式

| 方式 | 语法 | 返回类型 | 适用场景 |
|------|------|----------|----------|
| 直接调用 | `model(x)` | tf.Tensor | 自定义训练循环、需要梯度 |
| predict() | `model.predict(x)` | NumPy 数组 | 批量推理、部署 |

### 使用建议

- 训练时用 `model(x)`：因为可能需要梯度
- 预测时用 `predict()`：更方便，返回 NumPy 数组直接可用

In [ ]:
# =========================
# Cell 25：直接调用模型进行推理
# =========================
#
# model(x) 返回的是 TensorFlow Tensor
# 可以继续参与计算图运算（可以求梯度）
#

# 取验证集前 10 个样本做测试
new_inputs = val_inputs[:10]

# 直接调用模型（就像调用函数）
pred_tensor = binary_model_with_val(new_inputs)

print(f"model(x) 返回类型：{type(pred_tensor)}")
print(f"预测值形状：{pred_tensor.shape}")
print(f"\n前 5 个预测值（概率）：")
for i, p in enumerate(pred_tensor[:5].numpy().flatten()):
    print(f"  样本 {i+1}: {p:.4f} → 预测类别 {int(p > 0.5)}")

print(f"\n对应的真实标签：{val_targets[:5].flatten()}")

In [ ]:
# =========================
# Cell 26：predict() 批量推理
# =========================
#
# predict() 更适合实际部署场景：
# - 返回 NumPy 数组，方便后续处理
# - 支持大数据集分批处理
# - 不需要梯度，效率更高
#

pred_np = binary_model_with_val.predict(new_inputs, batch_size=128, verbose=0)

print(f"predict() 返回类型：{type(pred_np)}")
print(f"预测值形状：{pred_np.shape}")
print(f"\n前 10 个预测结果：")
print(f"  概率      →  预测类别  |  真实标签")
print("-" * 40)
for i, (p, t) in enumerate(zip(pred_np.flatten(), val_targets[:10].flatten())):
    pred_class = int(p > 0.5)
    match = "✓" if pred_class == int(t) else "✗"
    print(f"  {p:.4f}   →     {pred_class}     |     {int(t)}     {match}")


### 什么时候用哪一个？

- **训练 / 自定义流程 / 需要 Tensor 运算**：常用 `model(x)`
- **普通批量预测 / 部署前测试**：常用 `model.predict()`

这就是截图中“`predict()` 常用于小批量推理，并返回 NumPy 数组”的核心意思。


---
## Cell 27：多分类场景 — 3 种损失函数对比

### 分类问题的损失函数选择

| 任务类型 | 标签格式 | 损失函数 |
|----------|----------|----------|
| 二分类 | 0/1 整数 | `BinaryCrossentropy` |
| 多分类 | 整数标签 (0,1,2,...) | `SparseCategoricalCrossentropy` |
| 多分类 | One-hot 编码 | `CategoricalCrossentropy` |

### SparseCategorical vs Categorical 区别

```python
# 标签是整数：用 SparseCategoricalCrossentropy
y = [0, 2, 1, 0]  # 类别编号

# 标签是 one-hot：用 CategoricalCrossentropy
y = [[1,0,0], [0,0,1], [0,1,0], [1,0,0]]  # one-hot 编码
```

**推荐**：用 `SparseCategoricalCrossentropy`，标签更简洁。

In [ ]:
# =========================
# Cell 28：生成多分类数据集（3 类）
# =========================

np.random.seed(123)

num_samples_mc = 1500   # 样本数
num_features_mc = 16    # 特征数
num_classes = 3         # 类别数

# 随机生成输入
x_mc = np.random.randn(num_samples_mc, num_features_mc).astype("float32")

# 随机构造分类边界
W_mc = np.random.randn(num_features_mc, num_classes).astype("float32")
logits_mc = x_mc @ W_mc

# 生成两种格式的标签
y_int = np.argmax(logits_mc, axis=1).astype("int32")  # 整数标签：[0, 2, 1, ...]
y_onehot = keras.utils.to_categorical(y_int, num_classes=num_classes).astype("float32")  # one-hot

print(f"输入数据形状：{x_mc.shape}")
print(f"\n整数标签形状：{y_int.shape}")
print(f"前 5 个整数标签：{y_int[:5]}")

print(f"\nOne-hot 标签形状：{y_onehot.shape}")
print(f"前 5 个 one-hot 标签：")
for i in range(5):
    print(f"  类别 {y_int[i]} → {y_onehot[i]}")

In [ ]:
# =========================
# Cell 29：SparseCategoricalCrossentropy 示例
# =========================
#
# 标签是整数 [0, 1, 2, ...]，不需要转成 one-hot
# 这是更常用的方式，更简洁
#

sparse_model = keras.Sequential([
    layers.Dense(32, activation="relu"),
    layers.Dense(num_classes, activation="softmax")  # 输出 3 个概率
], name="sparse_multiclass_model")

sparse_model.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(),    # 注意这里
    metrics=[keras.metrics.SparseCategoricalAccuracy()]   # 注意这里
)

print("=== SparseCategoricalCrossentropy ===")
print("输入标签格式：整数 [0, 1, 2, ...]")
print(f"标签形状：{y_int.shape}\n")

history_sparse = sparse_model.fit(
    x_mc, y_int,       # 直接传整数标签
    epochs=3,
    batch_size=32,
    verbose=1
)

In [ ]:
# =========================
# Cell 30：CategoricalCrossentropy 示例
# =========================
#
# 标签是 one-hot 编码 [[1,0,0], [0,1,0], [0,0,1]]
# 需要先用 to_categorical() 转换
#

categorical_model = keras.Sequential([
    layers.Dense(32, activation="relu"),
    layers.Dense(num_classes, activation="softmax")
], name="categorical_multiclass_model")

categorical_model.compile(
    optimizer="adam",
    loss=keras.losses.CategoricalCrossentropy(),      # 注意这里
    metrics=[keras.metrics.CategoricalAccuracy()]     # 注意这里
)

print("=== CategoricalCrossentropy ===")
print("输入标签格式：one-hot [[1,0,0], [0,1,0], [0,0,1]]")
print(f"标签形状：{y_onehot.shape}\n")

history_categorical = categorical_model.fit(
    x_mc, y_onehot,   # 传 one-hot 标签
    epochs=3,
    batch_size=32,
    verbose=1
)


### 两者区别

- `SparseCategoricalCrossentropy`
  - 标签形状通常是 `(batch,)`
  - 标签值如：`[0, 2, 1, 1, 0]`
- `CategoricalCrossentropy`
  - 标签形状通常是 `(batch, num_classes)`
  - 标签是 one-hot，如：
    - 类别 0 -> `[1, 0, 0]`
    - 类别 1 -> `[0, 1, 0]`
    - 类别 2 -> `[0, 0, 1]`


---
## Cell 31：回归场景 — `MeanSquaredError`

### 回归 vs 分类

| | 分类 | 回归 |
|---|------|------|
| 输出 | 类别概率 | 连续数值 |
| 激活函数 | softmax/sigmoid | 无（或 linear） |
| 损失函数 | CrossEntropy | MSE/MAE |
| 例子 | 图像分类、情感分析 | 房价预测、温度预测 |

### MSE 公式

$$\text{MSE} = \frac{1}{N}\sum_{i=1}^{N}(y_i - \hat{y}_i)^2$$

就是"预测值和真实值差多少，平方后取平均"。

In [ ]:
# =========================
# Cell 32：生成回归数据集
# =========================
#
# 回归任务：预测连续数值
# 比如：根据房屋面积、房间数、位置等预测房价
#

np.random.seed(7)

num_samples_reg = 1200
num_features_reg = 10

# 输入特征
x_reg = np.random.randn(num_samples_reg, num_features_reg).astype("float32")

# 真实的关系：y = X @ W + 噪声
true_w_reg = np.random.randn(num_features_reg, 1).astype("float32")
y_reg = x_reg @ true_w_reg + 0.1 * np.random.randn(num_samples_reg, 1).astype("float32")

print(f"输入形状：{x_reg.shape}")
print(f"标签形状：{y_reg.shape}")
print(f"\n这是一个典型的回归问题：")
print(f"  输入：{num_features_reg} 个特征")
print(f"  输出：1 个连续数值")
print(f"\n前 5 个标签值：{y_reg[:5].flatten()}")

In [ ]:
# =========================
# Cell 33：回归模型 + MSE 损失
# =========================
#
# 回归模型的特点：
# - 最后一层没有激活函数（输出任意实数）
# - 用 MSE 或 MAE 作为损失函数
#

reg_model = keras.Sequential([
    layers.Dense(32, activation="relu"),
    layers.Dense(1)  # 注意：没有激活函数！输出可以是任意实数
], name="regression_model")

reg_model.compile(
    optimizer="rmsprop",
    loss=keras.losses.MeanSquaredError(),           # MSE 损失
    metrics=[keras.metrics.MeanAbsoluteError()]     # MAE 指标
)

print("=== 回归模型训练 ===")
print("损失函数：MSE（均方误差）")
print("监控指标：MAE（平均绝对误差）\n")

history_reg = reg_model.fit(
    x_reg, y_reg,
    epochs=5,
    batch_size=32,
    verbose=1
)

# 预测几个样本
predictions = reg_model.predict(x_reg[:5], verbose=0)
print("\n预测示例：")
print(f"  真实值      预测值       误差")
print("-" * 40)
for true, pred in zip(y_reg[:5], predictions[:5]):
    print(f"  {true[0]:.4f}    {pred[0]:.4f}    {abs(true[0]-pred[0]):.4f}")


## 20. 其他截图中提到的损失函数和指标：快速演示

截图列出了很多常见对象，例如：

### 优化器
- `SGD`
- `RMSprop`
- `Adam`
- `Adagrad`

### 损失函数
- `CategoricalCrossentropy`
- `SparseCategoricalCrossentropy`
- `BinaryCrossentropy`
- `MeanSquaredError`
- `KLDivergence`
- `CosineSimilarity`

### 指标
- `CategoricalAccuracy`
- `SparseCategoricalAccuracy`
- `BinaryAccuracy`

下面演示这些对象如何实例化。


In [ ]:

# =========================
# 29. 常见优化器 / 损失 / 指标对象
# =========================
optimizers_demo = {
    "SGD": keras.optimizers.SGD(learning_rate=1e-2),
    "RMSprop": keras.optimizers.RMSprop(learning_rate=1e-3),
    "Adam": keras.optimizers.Adam(learning_rate=1e-3),
    "Adagrad": keras.optimizers.Adagrad(learning_rate=1e-2),
}

losses_demo = {
    "BinaryCrossentropy": keras.losses.BinaryCrossentropy(),
    "CategoricalCrossentropy": keras.losses.CategoricalCrossentropy(),
    "SparseCategoricalCrossentropy": keras.losses.SparseCategoricalCrossentropy(),
    "MeanSquaredError": keras.losses.MeanSquaredError(),
    "KLDivergence": keras.losses.KLDivergence(),
    "CosineSimilarity": keras.losses.CosineSimilarity(),
}

metrics_demo = {
    "BinaryAccuracy": keras.metrics.BinaryAccuracy(),
    "CategoricalAccuracy": keras.metrics.CategoricalAccuracy(),
    "SparseCategoricalAccuracy": keras.metrics.SparseCategoricalAccuracy(),
}

print("=== 优化器示例 ===")
for name, obj in optimizers_demo.items():
    print(name, "->", obj)

print("\n=== 损失函数示例 ===")
for name, obj in losses_demo.items():
    print(name, "->", obj)

print("\n=== 指标示例 ===")
for name, obj in metrics_demo.items():
    print(name, "->", obj)



## 21. `KLDivergence` 与 `CosineSimilarity` 的小例子

这两个在入门模型里不一定最常用，但截图确实提到了，这里也补上：

- **KLDivergence**
  - 常用于比较两个概率分布
- **CosineSimilarity**
  - 常用于比较向量方向相似性


In [ ]:

# =========================
# 30. KLDivergence 示例
# =========================
y_true_dist = tf.constant([[0.7, 0.2, 0.1],
                           [0.1, 0.3, 0.6]], dtype=tf.float32)

y_pred_dist = tf.constant([[0.6, 0.3, 0.1],
                           [0.2, 0.2, 0.6]], dtype=tf.float32)

kld = keras.losses.KLDivergence()
kld_value = kld(y_true_dist, y_pred_dist)

print("KLDivergence:", float(kld_value))


In [ ]:

# =========================
# 31. CosineSimilarity 示例
# =========================
vec_a = tf.constant([[1.0, 2.0, 3.0],
                     [1.0, 0.0, 0.0]], dtype=tf.float32)

vec_b = tf.constant([[1.0, 2.0, 2.5],
                     [0.0, 1.0, 0.0]], dtype=tf.float32)

cosine_loss = keras.losses.CosineSimilarity(axis=-1)
cosine_value = cosine_loss(vec_a, vec_b)

print("CosineSimilarity 返回的是“负余弦相似度”的均值：")
print(cosine_value.numpy())



## 22. `model.summary()` 的意义

无论是 `Sequential` 还是函数式模型，`summary()` 都很重要，因为它能帮助你快速确认：

- 每一层的名字
- 每一层的输出形状
- 参数量（Param #）
- 总参数量

这在你调模型结构时非常重要。


In [ ]:

# =========================
# 32. 再看一次不同模型的 summary
# =========================
print("=== functional_mlp ===")
functional_mlp.summary()

print("\n=== binary_model_with_val ===")
binary_model_with_val.summary()


---
## 总结：Keras API 全流程回顾

### 核心概念链

```
┌─────────────────────────────────────────────────────────────┐
│  1. 层（Layer）                                              │
│     - 封装状态（权重）和计算（前向传播）                       │
│     - build() 创建参数，call() 定义计算                       │
│     - 支持自动形状推断                                        │
└─────────────────────────────────────────────────────────────┘
                              ↓
┌─────────────────────────────────────────────────────────────┐
│  2. 模型（Model）                                            │
│     - Sequential：简单的线性堆叠                              │
│     - 函数式 API：复杂的拓扑结构（残差、多输入输出）            │
└─────────────────────────────────────────────────────────────┘
                              ↓
┌─────────────────────────────────────────────────────────────┐
│  3. 编译（compile）                                          │
│     - optimizer：优化器（Adam, SGD, RMSprop）                 │
│     - loss：损失函数（MSE, CrossEntropy）                     │
│     - metrics：监控指标（Accuracy, MAE）                      │
└─────────────────────────────────────────────────────────────┘
                              ↓
┌─────────────────────────────────────────────────────────────┐
│  4. 训练（fit）                                              │
│     - 输入数据和标签                                          │
│     - epochs：训练轮数                                        │
│     - batch_size：批大小                                      │
│     - validation_data：验证集                                 │
└─────────────────────────────────────────────────────────────┘
                              ↓
┌─────────────────────────────────────────────────────────────┐
│  5. 评估与推理                                               │
│     - evaluate()：在测试集上评估                              │
│     - predict()：对新数据预测                                 │
└─────────────────────────────────────────────────────────────┘
```

### 任务类型与损失函数对照表

| 任务 | 输出层 | 损失函数 | 标签格式 |
|------|--------|----------|----------|
| 二分类 | `Dense(1, sigmoid)` | `BinaryCrossentropy` | 0/1 |
| 多分类 | `Dense(n, softmax)` | `SparseCategoricalCrossentropy` | 整数 |
| 多分类 | `Dense(n, softmax)` | `CategoricalCrossentropy` | one-hot |
| 回归 | `Dense(1)` | `MeanSquaredError` | 连续值 |

### 常用 API 速查

```python
# 1. 创建模型
model = keras.Sequential([layers.Dense(32), layers.Dense(10)])

# 2. 编译
model.compile(optimizer="adam", loss="mse", metrics=["mae"])

# 3. 训练
history = model.fit(x, y, epochs=10, batch_size=32, validation_split=0.2)

# 4. 评估
loss, mae = model.evaluate(x_test, y_test)

# 5. 预测
predictions = model.predict(x_new)
```

---

这个 Notebook 覆盖了 Keras 的核心 API，可以直接作为学习和汇报的基础材料。